In [0]:
-- League - Home Ice Advantage
-- Win % at home vs on the road for every team.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    CAST(NULL AS STRING) AS team,  -- team abbreviation; CAST(NULL AS STRING) = every team
    20252026 AS season,            -- e.g. 20242025; CAST(NULL AS INT) = every season
    2 AS game_type                 -- 2 = regular season, 3 = playoffs
),
team_games AS (                          -- one row per team per finished game
  SELECT g.game_id, g.season, g.game_date, g.date_key, g.last_period_type,
         t.team_id, t.abbrev AS team, o.abbrev AS opponent,
         t.team_id = g.home_team_id AS is_home, g.home_team_id,
         COALESCE(t.abbrev = pr.team, FALSE) AS is_selected,
         IF(t.team_id = g.home_team_id, g.home_final_score, g.away_final_score) AS final_gf,
         IF(t.team_id = g.home_team_id, g.away_final_score, g.home_final_score) AS final_ga,
         CASE WHEN IF(t.team_id = g.home_team_id, g.home_final_score > g.away_final_score,
                                                  g.away_final_score > g.home_final_score) THEN 'W'
              WHEN g.game_type = 2 AND g.last_period_type IN ('OT', 'SO') THEN 'OTL'
              ELSE 'L' END AS result
  FROM nhl.gold.dim_game g
  JOIN nhl.gold.dim_team t ON t.team_id IN (g.home_team_id, g.away_team_id)
  JOIN nhl.gold.dim_team o ON o.team_id = IF(t.team_id = g.home_team_id, g.away_team_id, g.home_team_id)
  CROSS JOIN params pr
  WHERE g.is_final
    AND g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
    AND (pr.team IS NULL OR t.abbrev = pr.team)
)
SELECT team,
       COUNT_IF(is_home)                                                        AS home_games,
       ROUND(100.0 * COUNT_IF(is_home AND result = 'W') / NULLIF(COUNT_IF(is_home), 0), 1)         AS home_win_pct,
       COUNT_IF(NOT is_home)                                                    AS away_games,
       ROUND(100.0 * COUNT_IF(NOT is_home AND result = 'W') / NULLIF(COUNT_IF(NOT is_home), 0), 1) AS away_win_pct,
       ROUND(100.0 * COUNT_IF(is_home AND result = 'W') / NULLIF(COUNT_IF(is_home), 0)
           - 100.0 * COUNT_IF(NOT is_home AND result = 'W') / NULLIF(COUNT_IF(NOT is_home), 0), 1) AS home_edge
FROM team_games
GROUP BY team
ORDER BY home_edge DESC;